## Import

In [1]:
from bertopic import BERTopic
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

c:\Users\aleblu\AppData\Local\miniconda3\envs\NLP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])
print(docs[:5])
classes = list(df["gen"])

['Consider each round carefully as it appears the same colours always go up against each other.\xa0 So yellow will always play against orange and purple against green.\xa0 Choose each colour in each round and remember who is best.\xa0 This will maximise your number of points.\xa0 Good luck.\xa0\xa0', 'If the multiplier is 5, then choose the pointy hat (if they have different hats)If the multiplier is 1, then choose the round hat (if they have different hats)If they have the same hats and you have a choice of pink and brown, choose brown.\xa0If they have the same hats and you have a choice of yellow and red, you need to work out what colour the baske will be and choose that colour. That will alternate between red and yellow each go, so try and take notegood luck!', 'The game is fantastic but requires you to think quickly and react.The features and characters is easy to use and move as well as the keys to play ths game is easy to each without hurting the fingers or hand.A lovely game. ',

## Pre-caculate Embeddings

Shorten run time once calculated

In [3]:
# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 32/32 [00:13<00:00,  2.43it/s]


## Preventing Stochastic Behavior
Reduce dimenstion (size of embeddings).

Also allows for reproduction every time the model is run.

n_neighbors: number of neighboring sample points used when making the manifold approximation. Increase -> more global view; Decrease -> more local view.

n_components: dimensionality of the embeddings after reducing them. Change clustering from HDBSCAN.

In [24]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=314346)

## Controlling Number of Topics
Using HDBSCAN, we can merge topics **after** creation.

min_cluster_size: controls the minimum size of a cluster and thereby the number of clusters that will be generated. High -> fewer larger clusters; Low -> more micro clusters.

In [5]:
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

## Improving default representation
- Remove stopwords
- Ignore infrequent words
- Increase the n-gram range (to 2) 

In [6]:
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

## Training

In [25]:
topic_model = BERTopic(

    # Pipeline models
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,

    # Hyperparameters
    top_n_words=10,
    n_gram_range=(1, 2),
    min_topic_size="auto", #use HDBSCAN
    verbose=True,

    # General parameters
    calculate_probabilities=True,
    language="english"
)

# Train model
topics, probs = topic_model.fit_transform(docs, embeddings)
# topic_per_class = topic_model.topics_per_class(docs, classes=classes)
# topics_over_time = topic_model.topics_over_time(docs, classes=classes)

# Show topics
topic_model.get_topic_info()

2025-01-29 17:49:12,176 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-01-29 17:49:14,227 - BERTopic - Dimensionality - Completed ✓
2025-01-29 17:49:14,227 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-01-29 17:49:14,281 - BERTopic - Cluster - Completed ✓
2025-01-29 17:49:14,281 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-01-29 17:49:14,360 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,166,-1_gnome_gnomes_mushrooms_colour,"[gnome, gnomes, mushrooms, colour, time, brown...",[Use the first few decisions on each round to ...
1,0,206,0_gnome_points_gnomes_hat,"[gnome, points, gnomes, hat, colour, brown, co...",[Different coloured Gnomes will provide points...
2,1,158,1_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...",[There are eight gnomes of different colours (...
3,2,106,2_mushrooms_gnome_gnomes_try,"[mushrooms, gnome, gnomes, try, colour, change...",[It might be difficult choosing the gnome that...
4,3,84,3_basket_red_baskets_yellow,"[basket, red, baskets, yellow, points, gnomes,...",[On the screen you will be presented with two ...
5,4,81,4_mushrooms_colours_mushroom_try,"[mushrooms, colours, mushroom, try, make, give...",[Try taking note of the number of mushrooms ea...
6,5,33,5_points_colours_scores_pink,"[points, colours, scores, pink, choose, colour...","[There are two themes of colours, purple, pink..."
7,6,33,6_hats_hat_tall_short,"[hats, hat, tall, short, taller, tall hats, go...","[Do your best, seems pretty random and difficu..."
8,7,29,7_keys_just_breaks_game,"[keys, just, breaks, game, fingers, make, hand...","[As each gnome appears, tap S for the left gno..."
9,8,23,8_hat_mushrooms_hats_tall,"[hat, mushrooms, hats, tall, modifier, gnome, ...",[I found that the gnome with the tall hat give...


## Evaluation

from octis

In [26]:
from octis.evaluation_metrics.diversity_metrics import TopicDiversity
from octis.evaluation_metrics.coherence_metrics import Coherence

# Retrieve topics (top_n_terms) from topic_model
dictionnary = topic_model.get_topics()
top_n_terms = []
for key in dictionnary:
    top_n_terms.append([i[0] for i in dictionnary[key]])

# Create model_output as seen in octis lib from BERTopic lib
model_output = {
    "topics": top_n_terms,
    "topic-word-matrix": topic_model.c_tf_idf_,
    "topic-document-matrix": topic_model.approximate_distribution(docs)}

# Create dataset (texts) as seen in octis lib from vectorizer + tokenizer
vectorizer = topic_model.vectorizer_model
tokenizer = vectorizer.build_tokenizer()
tokens = [tokenizer(doc) for doc in docs]


topic_diversity = TopicDiversity(topk=10)
topic_diversity_score = topic_diversity.score(model_output)

coherence = Coherence(texts=tokens, topk=10, measure="c_npmi")
coherence_score = coherence.score(model_output)

(topic_diversity_score, coherence_score)

100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


(0.5142857142857142, 0.01673712692394313)

Mock test of 5 seeds:

(0.5210526315789473, 0.013782963181978209)

(0.5210526315789473, 0.013782963181978209)

(0.49333333333333335, 0.01599178169451398)

(0.55, 0.020980773124476486)

(0.5142857142857142, 0.01673712692394313)

Not a lot of change

## Vizualisation

In [9]:
topic_model.visualize_topics()

In [10]:
topic_model.visualize_distribution(probs[10], min_probability=0.015)

In [11]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [12]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)

100%|██████████| 17/17 [00:00<00:00, 380.73it/s]

.
├─basket_red_yellow_gnomes_red basket
│    ├─■──forest_gnomes forest_mushrooms_green_green yellow ── Topic: 10
│    └─basket_red_yellow_gnomes_red basket
│         ├─basket_red_yellow_gnomes_red basket
│         │    ├─■──groups_score_scores_group_higher ── Topic: 13
│         │    └─basket_red_yellow_gnomes_red basket
│         │         ├─■──basket_red_yellow_mushrooms_red basket ── Topic: 0
│         │         └─■──basket_red_baskets_yellow_points ── Topic: 2
│         └─■──purple_blue_green_pink_brown ── Topic: 9
└─gnome_mushrooms_points_gnomes_hat
     ├─hat_hats_tall_short_gnome
     │    ├─■──hat_mushrooms_hats_modifier_tall ── Topic: 11
     │    └─hat_hats_tall_short_gnome
     │         ├─■──hat_gnome_hats_short_tall ── Topic: 4
     │         └─■──hats_tall_hat_tall hats_taller ── Topic: 6
     └─mushrooms_gnome_points_gnomes_colour
          ├─mushrooms_gnome_points_gnomes_colour
          │    ├─gnome_rounds_round_paying_use
          │    │    ├─■──paying_round_rounds_g

In [13]:
topic_model.visualize_documents(docs)

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
topic_model.get_document_info(docs)

In [ ]:
topic_model.visualize_topics_per_class(topic_per_class)

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time)

In [ ]:
topic_model.get_topic(7)

## Data saving

In [83]:
import csv

with open(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_topic_value.csv"), "w", newline="") as file:
    write = csv.writer(file)
    for topic in topics:
        write.writerow([topic])

## Topic reduction

In [ ]:
a = topic_model

a.reduce_topics(docs, nr_topics = 5)

a.get_topic_info()

In [ ]:
a.visualize_heatmap()

## TEST

In [27]:
from octis.models.LDA import LDA
from octis.dataset.dataset import Dataset
from octis.evaluation_metrics.diversity_metrics import TopicDiversity
from octis.evaluation_metrics.coherence_metrics import Coherence

In [28]:
# Define dataset
dataset = Dataset()
dataset.fetch_dataset("20NewsGroup")

In [29]:
# Create Model
model = LDA(num_topics=20, alpha=0.1)

In [30]:
# Train the model using default partitioning choice 
output = model.train_model(dataset)

print(*list(output.keys()), sep="\n") # Print the output identifiers

topic-word-matrix
topics
topic-document-matrix
test-topic-document-matrix


Recreate OCTIS output from BERTopic

+ topic-word-matrix
+ [SOLVED] topics
+ [SOLVED] topic-document-matrix
+ [SOLVED] texts

topic-word-matrix

In [134]:
output["topic-word-matrix"][0]

array([1.2224079e-03, 1.1098323e-03, 1.3412192e-02, ..., 3.4851948e-06,
       2.7240202e-04, 8.5215343e-06], dtype=float32)

In [118]:
len(output["topic-word-matrix"][0])

1612

In [141]:
len(topic_model.vectorizer_model.get_feature_names())

3994

In [143]:
print(topic_model.c_tf_idf_)

  (0, 3993)	0.0009974044468629482
  (0, 3992)	0.0009974044468629482
  (0, 3988)	0.0027717158636899777
  (0, 3986)	0.0019948088937258965
  (0, 3985)	0.0010557814542160365
  (0, 3984)	0.0010557814542160365
  (0, 3983)	0.0008393867967798322
  (0, 3980)	0.00640902938352943
  (0, 3977)	0.0010557814542160365
  (0, 3976)	0.0008755047017190772
  (0, 3975)	0.0016787735935596645
  (0, 3974)	0.0009559990429195514
  (0, 3972)	0.0010557814542160365
  (0, 3971)	0.0009974044468629482
  (0, 3970)	0.00537378120364144
  (0, 3965)	0.0017510094034381544
  (0, 3964)	0.0019948088937258965
  (0, 3963)	0.0010557814542160365
  (0, 3962)	0.0009974044468629482
  (0, 3960)	0.002503634248907023
  (0, 3959)	0.0010557814542160365
  (0, 3958)	0.0008242524173785896
  (0, 3957)	0.0008105661976827521
  (0, 3956)	0.005591357821423369
  (0, 3955)	0.0009974044468629482
  :	:
  (13, 174)	0.009627241809775883
  (13, 173)	0.008664568934947221
  (13, 167)	0.00722110706235691
  (13, 152)	0.005164912471076923
  (13, 140)	0.00962

topic_document-matrix

In [113]:
output["topic-document-matrix"]

array([[0.01428959, 0.76247495, 0.0022733 , ..., 0.01111398, 0.00188739,
        0.00178642],
       [0.01428619, 0.01250179, 0.00227329, ..., 0.01111417, 0.00188739,
        0.00178622],
       [0.01428598, 0.01250056, 0.00227389, ..., 0.01111353, 0.00188743,
        0.00178642],
       ...,
       [0.01428601, 0.01250066, 0.00227374, ..., 0.01111431, 0.00188765,
        0.00178651],
       [0.01428605, 0.01250082, 0.00227353, ..., 0.01111386, 0.00188752,
        0.00178635],
       [0.01428602, 0.01250071, 0.00227415, ..., 0.01111482, 0.00188747,
        0.00178631]])

In [116]:
topic_document_matrix, _ = topic_model.approximate_distribution(docs)
topic_document_matrix

100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


array([[0.11358247, 0.11092292, 0.03240436, ..., 0.12871485, 0.02957641,
        0.09464819],
       [0.08650018, 0.10637122, 0.05444197, ..., 0.05735415, 0.02290784,
        0.04294271],
       [0.        , 0.        , 0.        , ..., 0.        , 0.12174366,
        0.        ],
       ...,
       [0.06122985, 0.27464893, 0.06312755, ..., 0.08439809, 0.00545754,
        0.04707172],
       [0.04932114, 0.28507428, 0.        , ..., 0.42406742, 0.        ,
        0.        ],
       [0.06328944, 0.22275867, 0.09859579, ..., 0.07676761, 0.        ,
        0.04626213]])

TOPICS

In [76]:
output["topics"]

[['system',
  'file',
  'mail',
  'access',
  'work',
  'run',
  'address',
  'machine',
  'port',
  'information'],
 ['scsi',
  'drive',
  'device',
  'speed',
  'bit',
  'driver',
  'disk',
  'fast',
  'software',
  'memory'],
 ['armenian',
  'people',
  'turkish',
  'kill',
  'genocide',
  'village',
  'land',
  'government',
  'man',
  'dead'],
 ['drive', 'card', 'buy', 'make', 'work', 'car', 'good', 'bus', 'run', 'rate'],
 ['space',
  'launch',
  'power',
  'light',
  'period',
  'satellite',
  'year',
  'mission',
  'time',
  'orbit'],
 ['sell',
  'price',
  'sale',
  'offer',
  'good',
  'include',
  'mail',
  'card',
  'interested',
  'buy'],
 ['file',
  'window',
  'image',
  'program',
  'application',
  'version',
  'widget',
  'include',
  'run',
  'user'],
 ['state',
  'law',
  'weapon',
  'gun',
  'people',
  'jewish',
  'war',
  'arab',
  'military',
  'government'],
 ['game',
  'team',
  'win',
  'year',
  'play',
  'good',
  'player',
  'season',
  'make',
  'time'],
 

In [78]:
dictionnary = topic_model.get_topics()
top_n_terms = []
for key in dictionnary:
    top_n_terms.append([i[0] for i in dictionnary[key]])
top_n_terms

[['gnomes',
  'mushrooms',
  'gnome',
  'colour',
  'multiplier',
  'yellow',
  'red',
  'brown',
  'time',
  'colours'],
 ['gnome',
  'points',
  'gnomes',
  'hat',
  'colour',
  'brown',
  'left',
  'blue',
  'right',
  'colours'],
 ['basket',
  'red',
  'yellow',
  'mushrooms',
  'gnomes',
  'red basket',
  'yellow basket',
  'baskets',
  'blue',
  'gnome'],
 ['mushrooms',
  'gnome',
  'gnomes',
  'try',
  'colour',
  'change',
  'pair',
  'gives',
  'higher',
  'color'],
 ['basket',
  'red',
  'baskets',
  'yellow',
  'points',
  'gnomes',
  'colour',
  'red basket',
  'gnome',
  'colours'],
 ['mushrooms',
  'colours',
  'mushroom',
  'make',
  'color',
  'change',
  'try',
  'gives',
  'breaks',
  'game'],
 ['hats',
  'hat',
  'tall',
  'short',
  'better',
  'taller',
  'tall hats',
  'try',
  'good',
  'results'],
 ['points',
  'blue',
  'colours',
  'choices',
  'time',
  'choose',
  'colour',
  'pink',
  'change',
  'make'],
 ['keys',
  'just',
  'fingers',
  'breaks',
  'game

TEXTS

In [39]:
dataset.get_corpus()

[['fax', 'modem', 'card', 'sell', 'mail'],
 ['run', 'server', 'server', 'install', 'run', 'add'],
 ['live',
  'part',
  'lead',
  'wait',
  'important',
  'remember',
  'judge',
  'judge',
  'guess',
  'close',
  'situation',
  'listen',
  'statement',
  'sense',
  'regard',
  'passage',
  'remember',
  'letter',
  'church',
  'people',
  'body',
  'talk',
  'work',
  'translation',
  'lack',
  'concern',
  'make',
  'sick',
  'point',
  'throw',
  'faith',
  'faith',
  'catch',
  'meaning',
  'offer',
  'explanation',
  'fire',
  'cold',
  'make',
  'aware',
  'child',
  'eternal'],
 ['doesn', 'pain', 'deserve', 'die', 'lie', 'rape'],
 ['sale',
  'mile',
  'good',
  'condition',
  'good',
  'condition',
  'player',
  'component',
  'speaker',
  'mount',
  'door',
  'car',
  'maintain',
  'clean',
  'good',
  'car',
  'solid',
  'body',
  'spot',
  'surface',
  'spot',
  'touch',
  'make',
  'car',
  'problem',
  'firm',
  'car',
  'average',
  'cost',
  'interested',
  'call',
  'emai

In [36]:
tokenizer = vectorizer.build_tokenizer()
texts = [tokenizer(doc) for doc in docs]
texts

[['Consider',
  'each',
  'round',
  'carefully',
  'as',
  'it',
  'appears',
  'the',
  'same',
  'colours',
  'always',
  'go',
  'up',
  'against',
  'each',
  'other',
  'So',
  'yellow',
  'will',
  'always',
  'play',
  'against',
  'orange',
  'and',
  'purple',
  'against',
  'green',
  'Choose',
  'each',
  'colour',
  'in',
  'each',
  'round',
  'and',
  'remember',
  'who',
  'is',
  'best',
  'This',
  'will',
  'maximise',
  'your',
  'number',
  'of',
  'points',
  'Good',
  'luck'],
 ['If',
  'the',
  'multiplier',
  'is',
  'then',
  'choose',
  'the',
  'pointy',
  'hat',
  'if',
  'they',
  'have',
  'different',
  'hats',
  'If',
  'the',
  'multiplier',
  'is',
  'then',
  'choose',
  'the',
  'round',
  'hat',
  'if',
  'they',
  'have',
  'different',
  'hats',
  'If',
  'they',
  'have',
  'the',
  'same',
  'hats',
  'and',
  'you',
  'have',
  'choice',
  'of',
  'pink',
  'and',
  'brown',
  'choose',
  'brown',
  'If',
  'they',
  'have',
  'the',
  'same',

In [77]:
# Preprocess documents
cleaned_docs = topic_model._preprocess_text(docs)

# Extract vectorizer and tokenizer from BERTopic
vectorizer = topic_model.vectorizer_model
tokenizer = vectorizer.build_tokenizer()

tokens = [tokenizer(doc) for doc in cleaned_docs]

In [55]:
# Initialize metric
npmi = Coherence(texts=dataset.get_corpus(), topk=10, measure='c_npmi')

In [56]:
npmi_score = npmi.score(output)
print("Coherence: "+str(npmi_score))

Coherence: 0.04185992674792443


## MULTIPLE TOPICS PER DOC

In [ ]:
test = topic_model.approximate_distribution(docs)
a = test[0]